In [ ]:
# -------------------------------------------------------------
# Load libraries
# -------------------------------------------------------------
import pandas as pd
import numpy as np
import Levenshtein  # Using the C-optimized version

# Shortcut
lev = Levenshtein.distance

# -------------------------------------------------------------
# Load dataset (your folder structure)
# -------------------------------------------------------------
df = pd.read_csv("../1_Data/preprocessed_data.csv")

# Convert sequences to pure Python strings (faster)
vh_sequences = df["VH Protein"].astype(str).tolist()
vl_sequences = df["VL Protein"].astype(str).tolist()
clone_names = df["Clone name"].tolist()

n = len(df)
print(f"Loaded {n} clones.")

# -------------------------------------------------------------
# Prepare empty distance matrices
# -------------------------------------------------------------
vh_dist_matrix = np.zeros((n, n), dtype=int)
vl_dist_matrix = np.zeros((n, n), dtype=int)

# -------------------------------------------------------------
# Compute ONLY the upper triangle of the matrices
# -------------------------------------------------------------
for i in range(n):
    seq_i_vh = vh_sequences[i]
    seq_i_vl = vl_sequences[i]

    for j in range(i, n):
        # VH distance
        d_vh = lev(seq_i_vh, vh_sequences[j])
        vh_dist_matrix[i, j] = d_vh
        vh_dist_matrix[j, i] = d_vh  # mirror

        # VL distance
        d_vl = lev(seq_i_vl, vl_sequences[j])
        vl_dist_matrix[i, j] = d_vl
        vl_dist_matrix[j, i] = d_vl  # mirror

    if i % 25 == 0:
        print(f"Progress: {i}/{n} rows computed...")

print("Distance calculation completed!")

# -------------------------------------------------------------
# Convert to DataFrames for readability/usage
# -------------------------------------------------------------
vh_df = pd.DataFrame(vh_dist_matrix, index=clone_names, columns=clone_names)
vl_df = pd.DataFrame(vl_dist_matrix, index=clone_names, columns=clone_names)

vh_df.head(), vl_df.head()

In [ ]:
vh_df.to_csv("VH_levenshtein_matrix.csv")
vl_df.to_csv("VL_levenshtein_matrix.csv")

print("Matrices saved in folder.")

In [ ]:
import numpy as np
import pandas as pd

def normalize_matrix_by_maxlen(seq_list, dist_matrix):
    n = len(seq_list)
    norm = np.zeros_like(dist_matrix, dtype=float)
    lengths = [len(s) for s in seq_list]
    for i in range(n):
        for j in range(n):
            denom = max(lengths[i], lengths[j]) if max(lengths[i], lengths[j]) > 0 else 1
            norm[i, j] = dist_matrix[i, j] / denom
    return pd.DataFrame(norm, index=vh_df.index, columns=vh_df.columns)

vh_norm = normalize_matrix_by_maxlen(vh_sequences, vh_df.values)
vl_norm = normalize_matrix_by_maxlen(vl_sequences, vl_df.values)

In [ ]:
# Inspecting VH norm values

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.heatmap(vh_norm, cmap="viridis", xticklabels=False, yticklabels=False)
plt.title("VH normalized Levenshtein distance")
plt.show()

In [ ]:
import re
def clean(seq): 
    return re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', str(seq).upper())

vh_clean = [clean(s) for s in vh_sequences]

In [ ]:
import numpy as np

vals = []
for i in range(n):
    for j in range(i+1, n):
        vals.append(vh_norm.iloc[i, j])
vals = np.array(vals)
print("VH norm distance – mean:", vals.mean(), 
      "median:", np.median(vals), 
      "p10/p90:", np.percentile(vals, [10, 90]))

In [ ]:
import seaborn as sns
sns.clustermap(
    vh_norm, metric="euclidean", method="average",
    cmap="viridis", figsize=(10,10),
    xticklabels=False, yticklabels=False
)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# Your precomputed distance matrix as a NumPy array
D = vh_norm.values  # vh_norm is your DataFrame of normalized distances (n x n)

# Convert to condensed form (what linkage expects)
# checks=False skips redundant validation since we know D is a proper distance matrix
Dc = squareform(D, checks=False)

# Choose a linkage method; 'average' (UPGMA) is a good default for distances like these
Z = linkage(Dc, method="average")

# Use the same linkage for rows and columns (symmetrical matrix)
g = sns.clustermap(
    vh_norm,
    row_linkage=Z,
    col_linkage=Z,
    cmap="viridis",
    xticklabels=False, yticklabels=False,
    figsize=(10, 10)
)
plt.show()

In [ ]:
# Inspecting VL norm values

In [ ]:
import re
def clean(seq): 
    return re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', str(seq).upper())

vl_clean = [clean(s) for s in vl_sequences]

In [ ]:
import numpy as np

vals = []
for i in range(n):
    for j in range(i+1, n):
        vals.append(vl_norm.iloc[i, j])
vals = np.array(vals)
print("VL norm distance – mean:", vals.mean(), 
      "median:", np.median(vals), 
      "p10/p90:", np.percentile(vals, [10, 90]))

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# Your precomputed distance matrix as a NumPy array
D = vl_norm.values  # vh_norm is your DataFrame of normalized distances (n x n)

# Convert to condensed form (what linkage expects)
# checks=False skips redundant validation since we know D is a proper distance matrix
Dc = squareform(D, checks=False)

# Choose a linkage method; 'average' (UPGMA) is a good default for distances like these
Z = linkage(Dc, method="average")

# Use the same linkage for rows and columns (symmetrical matrix)
g = sns.clustermap(
    vl_norm,
    row_linkage=Z,
    col_linkage=Z,
    cmap="viridis",
    xticklabels=False, yticklabels=False,
    figsize=(10, 10)
)
plt.show()